# 🎓 [vd17] Google Opal & Antigravity Cowork 마스터 실습
## 완주로컬푸드 오늘의 판매현황 실시간 EDA 및 1시간 주기 자동화 대시보드

> **기획/총괄**: (AX)창업기술 이한규 대표  
> **모듈**: `vd17_opal_agy_cowork` (Visual Design & AI Master Series)

---

### 📌 실습 개요
1. **Google Opal**: 공공데이터포털(data.go.kr) '완주로컬푸드 품목별 판매현황' API를 노코드/로우코드 AI 워크플로우로 연결
2. **Antigravity & Cowork**: 수집된 실시간 데이터를 판다스(Pandas)로 정제하고 통계적 지표를 자동 산출하는 EDA 엔진 구동
3. **고해상도 시각화**: 맑은고딕/Pretendard 기반 300 DPI 비즈니스 차트 4종 렌더링
4. **실시간 대시보드**: 1시간 주기로 최신 데이터를 자동 반영하는 글래스모피즘 웹 리포트 구축

### 1단계: 필수 모듈 및 라이브러리 임포트

In [1]:
import os
import sys
import datetime
import pandas as pd
import matplotlib.pyplot as plt

# 상위 src 폴더 모듈 경로 추가
sys.path.append(os.path.abspath('../src'))

from opal_connector import fetch_or_generate_sales, generate_opal_blueprint
from eda_engine import run_eda, generate_executive_markdown
from visualizer import generate_all_charts
from dashboard_generator import generate_html_dashboard

print("✅ 필수 모듈 로드 완료!")

✅ 필수 모듈 로드 완료!


### 2단계: 공공데이터 오픈API 호출 및 Opal Blueprint 생성

In [2]:
# API 키가 없어도 실시간 동적 시뮬레이터(Fallback Engine)가 자동 작동합니다.
df, csv_path, blueprint_path = fetch_or_generate_sales(output_dir="../result")

print(f"📊 수집된 레코드 수: {len(df)}건")
display(df.head(10))

[INFO] 공공데이터포털(data.go.kr) 오픈API 실시간 연동 시도...
[NOTICE] 인증키 활성화 동기화 대기 중 (data.go.kr 코드 30: 신규 발급 키 30분~2시간 동기화 지연)
[INFO] Opal Dynamic Real-Time Simulation Engine을 가동합니다. (무중단 파이프라인)
[저장 완료] 실시간 판매 데이터: ../result\latest_sales.csv (72건, 모드: DYNAMIC_SIMULATION)
[저장 완료] Google Opal Blueprint: ../result\opal_app_blueprint.json
📊 수집된 레코드 수: 72건


,판매일자,집계시간,판매매장명,품목카테고리,품목명,판매단위,단위가격,판매수량,판매금액
0,2026-09-15,17:00,완주로컬푸드 모악산점,과채류,구이 친환경 딸기,500g/팩,11400,119,1356600
1,2026-09-15,17:00,완주로컬푸드 모악산점,근채류,완주 생강,1kg/봉,10450,103,1076350
2,2026-09-15,17:00,완주로컬푸드 모악산점,과수류,이서 황금배,5kg/상자,32000,159,5088000
3,2026-09-15,17:00,완주로컬푸드 모악산점,과채류,삼례 싱싱토마토,2kg/상자,17600,232,4083200
4,2026-09-15,17:00,완주로컬푸드 모악산점,근채류,봉동 당근,1단/묶음,4275,88,376200
5,2026-09-15,17:00,완주로컬푸드 모악산점,버섯류,모악산 참표고버섯,1kg/상자,20900,67,1400300
6,2026-09-15,17:00,완주로컬푸드 모악산점,서류,완주 햇고구마,3kg/상자,15000,152,2280000
7,2026-09-15,17:00,완주로컬푸드 모악산점,가공식품,봉동 생강한과 세트,1세트,33250,56,1862000
8,2026-09-15,17:00,완주로컬푸드 모악산점,곡류,용진 친환경 쌀,10kg/포,38000,120,4560000
9,2026-09-15,17:00,완주로컬푸드 모악산점,가공식품,완주 곶감 세트,1세트,42000,226,9492000


### 3단계: 탐색적 데이터 분석(EDA) 및 핵심 KPI 산출

In [3]:
eda_results = run_eda(df)
summary = eda_results['summary']

print("=== [당일 완주로컬푸드 핵심 KPI 지표] ===")
print(f"• 총 누적 매출액: {summary['total_revenue']:,}원")
print(f"• 총 판매 수량: {summary['total_quantity']:,}개")
print(f"• 평균 객단가: {summary['avg_price']:,}원")
print(f"• 매출 1위 직매장: {summary['top_store_name']} ({summary['top_store_share']}% 점유)")
print(f"• 당일 베스트셀러: {summary['top_item_name']} ({summary['top_item_rev']:,}원)")

# 직매장별 실적 요약표
print("\n=== [직매장별 실적 현황] ===")
display(eda_results['store_agg'])

=== [당일 완주로컬푸드 핵심 KPI 지표] ===
• 총 누적 매출액: 168,793,550원
• 총 판매 수량: 8,113개
• 평균 객단가: 20,805원
• 매출 1위 직매장: 완주로컬푸드 혁신점 (21.0% 점유)
• 당일 베스트셀러: 완주 곶감 세트 (40,702,200원)

=== [직매장별 실적 현황] ===


,판매매장명,매출액,판매수량,취급품목수,점유율
5,완주로컬푸드 혁신점,35382275,1853,12,21.0
4,완주로컬푸드 해전점,33103850,1604,12,19.6
1,완주로컬푸드 모악산점,32333050,1469,12,19.2
2,완주로컬푸드 삼례점,25911025,1246,12,15.4
3,완주로컬푸드 용진점,23827550,1073,12,14.1
0,완주로컬푸드 둔산점,18235800,868,12,10.8


### 4단계: 300 DPI 고해상도 경영진 시각화 차트 4종 생성

In [4]:
charts = generate_all_charts(eda_results, "../result")

# 주피터 노트북 인라인 차트 미리보기
plt.figure(figsize=(10, 5))
img = plt.imread(charts['c1'])
plt.imshow(img)
plt.axis('off')
plt.title("품목별 매출 TOP 10 (300 DPI)", fontsize=14, pad=10)
plt.show()

[저장 완료] 4대 고해상도 경영진 차트 생성 완료: ../result\charts


C:\Users\note\AppData\Local\Temp\ipykernel_20632\3443049102.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 5단계: 실시간 반응형 글래스모피즘 대시보드 빌드

In [5]:
dashboard_path = os.path.abspath("../result/latest_report.html")
generate_html_dashboard(eda_results, charts, dashboard_path)

print(f"🎉 실시간 대시보드가 생성되었습니다!\n경로: {dashboard_path}")

[저장 완료] 실시간 글래스모피즘 대시보드: c:\Users\note\vd\vd17_opal_agy_cowork\result\latest_report.html
🎉 실시간 대시보드가 생성되었습니다!
경로: c:\Users\note\vd\vd17_opal_agy_cowork\result\latest_report.html
